In [1]:
# =============================================================================
# Cell 1
# IMPORTS AND CONFIGURATION
# =============================================================================
import openai
import anthropic
import json
import time
import configparser
import tiktoken
from typing import List, Dict, Tuple, Optional, Callable
from collections import Counter
import numpy as np
from dataclasses import dataclass, asdict
import google.generativeai as genai
from itertools import combinations
import random
from datetime import datetime
from pathlib import Path
import pickle
import traceback
import pandas as pd
from difflib import get_close_matches, SequenceMatcher
import os

# Directories
SAVE_DIR = "C:/Users/STSI/OneDrive - Skagerak Energi/06-NæringsPhD/Egne papers/State of the art/Data"
#LOAD_DIR= "C:/Users/STSI/OneDrive - Skagerak Energi/06-NæringsPhD/Egne papers/State of the art/Data"
output_dir = Path("software_analysis_final")
output_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"✓ Configuration loaded | Timestamp: {timestamp}")


c:\git_repos\Literature-search-and-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Configuration loaded | Timestamp: 20251219_132509


In [2]:
# =============================================================================
# Cell 2
# API CLIENT INITIALIZATION
# =============================================================================

def initialize_openai():
    """Initialize OpenAI client from config file"""
    config = configparser.ConfigParser()
    config.read('config_LLM.txt')
    api_key = config['LLM'].get('OPENAI_API_KEY')
    model_type = config['LLM'].get('MODEL_TYPE_adv', 'gpt-4o-mini')
    client = openai.OpenAI(api_key=api_key)
    return client, model_type

def initialize_anthropic():
    """Initialize Anthropic client from config file"""
    config = configparser.ConfigParser()
    config.read('config_LLM.txt')
    api_key = config['LLM'].get('ANTHROPIC_API_KEY')
    client = anthropic.Anthropic(api_key=api_key) if api_key else None
    return client

def initialize_google():
    """Initialize Google Gemini client from config file"""
    config = configparser.ConfigParser()
    config.read('config_LLM.txt')
    api_key = config['LLM'].get('GOOGLE_API_KEY')
    if api_key:
        genai.configure(api_key=api_key)
        return True
    return False

print("✓ API initialization functions defined")


✓ API initialization functions defined


In [3]:
# =============================================================================
# Cell 3
# TOKEN COUNTING AND COST TRACKING
# =============================================================================

def num_tokens_from_string(string: str, model_name: str) -> int:
    """Get token count with fallback for unsupported models"""
    try:
        encoding = tiktoken.encoding_for_model(model_name)
        return len(encoding.encode(string))
    except KeyError:
        if model_name.startswith('gpt-5'):
            encoding = tiktoken.get_encoding("o200k_base")
            return len(encoding.encode(string))
        elif model_name.startswith('gpt-4'):
            encoding = tiktoken.get_encoding("cl100k_base")
            return len(encoding.encode(string))
        elif model_name.startswith('claude'):
            return int(len(string) / 3.5)
        elif model_name.startswith('models/gemini') or model_name.startswith('gemini'):
            return int(len(string) / 4)
        else:
            return len(string) // 4

def count_tokens_in_messages(messages: List[Dict], model: str) -> int:
    """Count tokens in a list of messages"""
    total_tokens = 0
    for message in messages:
        if isinstance(message.get('content'), str):
            total_tokens += num_tokens_from_string(message['content'], model)
        total_tokens += 4
    total_tokens += 3
    return total_tokens

class CreditTracker:
    """Track API usage and costs across all models"""
    
    PRICING = {
        'gpt-4o': {'input': 1.25, 'output': 5.00},
        'gpt-4o-mini': {'input': 0.075, 'output': 0.30},
        'claude-3-haiku-20240307': {'input': 0.25, 'output': 1.25},
        'claude-3-5-haiku-20241022': {'input': 0.80, 'output': 4.00},
        'claude-3-5-sonnet-20241022': {'input': 3.00, 'output': 15.00},
        'claude-sonnet-4-20250514': {'input': 3.00, 'output': 15.00},
        'models/gemini-2.5-flash': {'input': 0.075, 'output': 0.30},
        'models/gemini-2.0-flash': {'input': 0.075, 'output': 0.30},
        'models/gemini-2.0-flash-001': {'input': 0.075, 'output': 0.30},
        'gemini-2.0-flash': {'input': 0.075, 'output': 0.30},
    }

    def __init__(self):
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_cached_tokens = 0
        self.total_cost = 0
        self.model_usage = {}
        self.call_count = 0

    def update(self, model: str, input_tokens: int, output_tokens: int, cached_tokens: int = 0):
        """Update usage statistics"""
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        self.total_cached_tokens += cached_tokens
        self.call_count += 1

        pricing = self.PRICING.get(model, {'input': 0.00015, 'output': 0.0006})
        
        input_cost = (input_tokens / 1_000_000) * pricing['input']
        output_cost = (output_tokens / 1_000_000) * pricing['output']
        call_cost = input_cost + output_cost
        self.total_cost += call_cost

        if model not in self.model_usage:
            self.model_usage[model] = {
                'calls': 0, 'input_tokens': 0, 'output_tokens': 0,
                'cached_tokens': 0, 'cost': 0
            }

        self.model_usage[model]['calls'] += 1
        self.model_usage[model]['input_tokens'] += input_tokens
        self.model_usage[model]['output_tokens'] += output_tokens
        self.model_usage[model]['cached_tokens'] += cached_tokens
        self.model_usage[model]['cost'] += call_cost

    def get_stats(self):
        """Get current statistics"""
        return {
            "total_calls": self.call_count,
            "total_input_tokens": self.total_input_tokens,
            "total_output_tokens": self.total_output_tokens,
            "total_tokens": self.total_input_tokens + self.total_output_tokens,
            "total_cost": round(self.total_cost, 4),
            "average_cost_per_call": round(self.total_cost / max(self.call_count, 1), 4),
            "model_breakdown": {
                model: {
                    'calls': stats['calls'],
                    'total_tokens': stats['input_tokens'] + stats['output_tokens'],
                    'cost': round(stats['cost'], 4)
                }
                for model, stats in self.model_usage.items()
            }
        }

    def print_summary(self):
        """Print formatted summary"""
        stats = self.get_stats()
        print("\n" + "="*60)
        print("API USAGE SUMMARY")
        print("="*60)
        print(f"Total API Calls: {stats['total_calls']}")
        print(f"Total Tokens: {stats['total_tokens']:,}")
        print(f"  - Input: {stats['total_input_tokens']:,}")
        print(f"  - Output: {stats['total_output_tokens']:,}")
        if self.total_cached_tokens > 0:
            print(f"  - Cached: {self.total_cached_tokens:,}")
        print(f"\nTotal Cost: ${stats['total_cost']:.4f}")
        print(f"Average Cost per Call: ${stats['average_cost_per_call']:.4f}")

        if self.model_usage:
            print("\nBreakdown by Model:")
            print("-" * 60)
            for model, breakdown in stats['model_breakdown'].items():
                print(f"  {model}:")
                print(f"    Calls: {breakdown['calls']}")
                print(f"    Tokens: {breakdown['total_tokens']:,}")
                print(f"    Cost: ${breakdown['cost']:.4f}")
        print("="*60 + "\n")

print("✓ Token counting and CreditTracker defined")


✓ Token counting and CreditTracker defined


In [4]:
# =============================================================================
# Cell 4
# DATA STRUCTURES
# =============================================================================

@dataclass
class AssessmentResult:
    """Single LLM assessment result"""
    software: str
    method: str
    rank: int
    reasoning: str
    sources: List[str]
    llm_provider: str
    input_tokens: int = 0
    output_tokens: int = 0

@dataclass
class ConsensusResult:
    """Consensus across multiple LLMs"""
    software: str
    method: str
    final_rank: int
    confidence: float
    individual_ranks: Dict[str, int]
    individual_reasoning: Dict[str, str]
    individual_sources: Dict[str, List[str]]
    agreement_level: str
    total_tokens: int = 0
    total_cost: float = 0.0

print("✓ Data structures defined")


✓ Data structures defined


In [5]:
# =============================================================================
# Cell 5
# SOFTWARE-METHOD ASSESSOR - CORE FUNCTIONALITY
# =============================================================================

class SoftwareMethodAssessor:
    """Main class for software-method assessment using multiple LLMs"""
    
    def __init__(self, use_config: bool = True, timeout: int = 180, max_retries: int = 3):
        """Initialize assessor with API clients"""
        if use_config:
            self.openai_client, self.default_model = initialize_openai()
            self.anthropic_client = initialize_anthropic()
            self.google_enabled = initialize_google()
        else:
            self.openai_client = None
            self.anthropic_client = None
            self.google_enabled = False
            self.default_model = "gpt-4o-mini"
        
        self.credit_tracker = CreditTracker()
        self.timeout = timeout
        self.max_retries = max_retries
        
        self.system_prompt = """You are a technical software assessment expert specialized in power systems analysis software.

Use this ranking scale:
0 = No support (method cannot be implemented at all)
1 = Limited possibility for implementation or extension (requires significant workarounds)
2 = Indirectly supported through APIs or extensions (requires external tools/plugins)
3 = Directly implemented (native feature in the software)

CRITICAL: You MUST search for and provide actual references. Your assessment must be based on real, verifiable sources.

For each assessment:
1. Search for scientific papers demonstrating implementation (IEEE Xplore, ScienceDirect, arXiv, Google Scholar)
2. Find official documentation from the software vendor or project website
3. Look for GitHub repositories with code examples or open-source implementations
4. Check API documentation or extension/plugin capabilities
5. Review user forums, technical blogs, Stack Overflow, or case studies

Return your response in VALID JSON format with this exact structure:
{
    "rank": <0-3>,
    "reasoning": "<detailed explanation citing specific sources by number, e.g., 'According to [1], PSS/E supports...'>",
    "sources": [
        "https://example.com/documentation - Official PSS/E Manual on OPF",
        "https://doi.org/10.1109/... - Paper title by Author et al.",
        "https://github.com/org/repo/file.py - Implementation example"
    ]
}

Each source must include both the URL and a brief description separated by ' - '.
Minimum 2 sources required for ranks 2-3, minimum 1 source for rank 1."""

    def calculate_confidence(self, ranks: List[int]) -> Tuple[float, str]:
        """Calculate confidence score from multiple assessments"""
        if not ranks:
            return 0.0, "no_data"
        
        rank_counts = Counter(ranks)
        most_common_count = rank_counts.most_common(1)[0][1]
        total_ranks = len(ranks)
        confidence = most_common_count / total_ranks
        
        if total_ranks == 1:
            agreement_level = "single_assessment"
        elif confidence == 1.0:
            agreement_level = "perfect_agreement"
        elif confidence >= 0.75:
            agreement_level = "strong_agreement"
        elif confidence >= 0.5:
            agreement_level = "moderate_agreement"
        else:
            agreement_level = "weak_agreement"
        
        return confidence, agreement_level

    def export_results(self, results: List[ConsensusResult], filename: str):
        """Export results to JSON file"""
        output_data = [asdict(result) for result in results]
        with open(filename, 'w') as f:
            json.dump(output_data, f, indent=2)
        print(f"\n✓ Results exported to {filename}")

print("✓ SoftwareMethodAssessor core defined")


✓ SoftwareMethodAssessor core defined


In [6]:
# =============================================================================
# Cell 6
# BATCH PROCESSING METHODS
# =============================================================================

def create_batch_assessment_prompt(self, batch_items: List[Tuple[str, str]], batch_size: int = None) -> str:
    """Create a structured prompt for batch assessment"""
    batch_size = batch_size or len(batch_items)
    
    items_text = ""
    for idx, (software, method) in enumerate(batch_items, 1):
        items_text += f"\n{idx}. Software: {software}\n   Method: {method}\n"
    
    prompt = f"""You must assess {len(batch_items)} software-method combinations independently.

CRITICAL INSTRUCTIONS:
- Treat each pair as completely independent
- Do NOT let one assessment influence another
- Provide the SAME quality of research and reasoning for ALL items
- Each assessment must have its own sources

Items to assess:{items_text}

For EACH item above, perform independent research and provide sources with URLs.

Return a JSON array with exactly {len(batch_items)} objects:
[
  {{
    "software": "<software name>",
    "method": "<method name>",
    "rank": <0-3>,
    "reasoning": "<detailed explanation citing sources [1], [2], etc.>",
    "sources": [
      "https://... - Description",
      "https://... - Description"
    ]
  }},
  ...
]

IMPORTANT: Return ONLY the JSON array, no other text."""
    
    return prompt

SoftwareMethodAssessor.create_batch_assessment_prompt = create_batch_assessment_prompt

print("✓ Batch prompt creation added")


✓ Batch prompt creation added


In [7]:
# =============================================================================
# Cell 7
# LLM-SPECIFIC ASSESSMENT METHODS
# =============================================================================

def assess_batch_with_openai(self, batch_items: List[Tuple[str, str]], 
                             model: str = None, debug: bool = False) -> List[AssessmentResult]:
    """Assess a batch of items with OpenAI"""
    if model is None:
        model = self.default_model
    
    try:
        prompt = self.create_batch_assessment_prompt(batch_items)
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": prompt}
        ]
        
        response = self.openai_client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.3,
            max_tokens=4096,
            timeout=self.timeout,
            response_format={"type": "json_object"}
        )
        
        usage = response.usage
        cached_tokens = 0
        if hasattr(usage, 'prompt_tokens_details') and usage.prompt_tokens_details:
            cached_tokens = getattr(usage.prompt_tokens_details, 'cached_tokens', 0)
        
        self.credit_tracker.update(
            model=model,
            input_tokens=usage.prompt_tokens,
            output_tokens=usage.completion_tokens,
            cached_tokens=cached_tokens
        )
        
        content = response.choices[0].message.content
        
        # Parse JSON
        parsed = json.loads(content)
        if isinstance(parsed, dict):
            if 'assessments' in parsed:
                results_data = parsed['assessments']
            elif 'results' in parsed:
                results_data = parsed['results']
            else:
                results_data = next(v for v in parsed.values() if isinstance(v, list))
        else:
            results_data = parsed
        
        # Convert to AssessmentResult objects
        assessment_results = []
        for item_data in results_data:
            rank = int(item_data.get("rank", 0))
            sources = item_data.get("sources", [])
            reasoning = item_data.get("reasoning", "")
            
            if rank > 0 and len(sources) == 0:
                rank = 0
                reasoning += " [Rank lowered to 0: no sources]"
            
            assessment_results.append(AssessmentResult(
                software=item_data.get("software", ""),
                method=item_data.get("method", ""),
                rank=rank,
                reasoning=reasoning,
                sources=sources,
                llm_provider=f"openai_{model}",
                input_tokens=usage.prompt_tokens // len(batch_items),
                output_tokens=usage.completion_tokens // len(batch_items)
            ))
        
        return assessment_results
        
    except Exception as e:
        print(f"  ERROR in OpenAI batch assessment: {e}")
        return []

def assess_batch_with_claude(self, batch_items: List[Tuple[str, str]], 
                             model: str = "claude-3-5-haiku-20241022", 
                             debug: bool = False) -> List[AssessmentResult]:
    """Assess a batch of items with Claude"""
    if not self.anthropic_client:
        return []
    
    try:
        prompt = self.create_batch_assessment_prompt(batch_items)
        
        response = self.anthropic_client.messages.create(
            model=model,
            max_tokens=4096,
            temperature=0.3,
            timeout=self.timeout,
            system=self.system_prompt,
            messages=[{"role": "user", "content": prompt}]
        )
        
        self.credit_tracker.update(
            model=model,
            input_tokens=response.usage.input_tokens,
            output_tokens=response.usage.output_tokens
        )
        
        content = response.content[0].text
        
        # Clean markdown code blocks
        content = content.strip()
        if content.startswith('```'):
            lines = content.split('\n')
            start_idx = 0
            end_idx = len(lines)
            for i, line in enumerate(lines):
                if line.strip().startswith('```'):
                    if start_idx == 0:
                        start_idx = i + 1
                    else:
                        end_idx = i
                        break
            content = '\n'.join(lines[start_idx:end_idx])
        
        # Parse JSON
        parsed = json.loads(content)
        if isinstance(parsed, dict):
            if 'assessments' in parsed:
                results_data = parsed['assessments']
            elif 'results' in parsed:
                results_data = parsed['results']
            else:
                results_data = next(v for v in parsed.values() if isinstance(v, list))
        else:
            results_data = parsed
        
        # Convert to AssessmentResult objects
        assessment_results = []
        for item_data in results_data:
            rank = int(item_data.get("rank", 0))
            sources = item_data.get("sources", [])
            reasoning = item_data.get("reasoning", "")
            
            if rank > 0 and len(sources) == 0:
                rank = 0
                reasoning += " [Rank lowered to 0: no sources]"
            
            assessment_results.append(AssessmentResult(
                software=item_data.get("software", ""),
                method=item_data.get("method", ""),
                rank=rank,
                reasoning=reasoning,
                sources=sources,
                llm_provider=f"claude_{model}",
                input_tokens=response.usage.input_tokens // len(batch_items),
                output_tokens=response.usage.output_tokens // len(batch_items)
            ))
        
        return assessment_results
        
    except Exception as e:
        print(f"  ERROR in Claude batch assessment: {e}")
        return []

def assess_batch_with_google(self, batch_items: List[Tuple[str, str]], 
                             model: str = "models/gemini-2.0-flash", 
                             debug: bool = False) -> List[AssessmentResult]:
    """Assess a batch of items with Google Gemini"""
    if not self.google_enabled:
        return []
    
    try:
        prompt = self.create_batch_assessment_prompt(batch_items)
        
        if not model.startswith('models/'):
            model = f"models/{model}"
        
        gemini_model = genai.GenerativeModel(
            model_name=model,
            generation_config={
                "temperature": 0.3,
                "max_output_tokens": 4096,
            },
            system_instruction=self.system_prompt
        )
        
        response = gemini_model.generate_content(
            prompt,
            generation_config={
                "temperature": 0.3,
                "max_output_tokens": 4096,
            },
            request_options={'timeout': self.timeout}
        )
        
        # Extract token counts
        try:
            input_tokens = response.usage_metadata.prompt_token_count
            output_tokens = response.usage_metadata.candidates_token_count
        except AttributeError:
            input_tokens = int(len(prompt.split()) * 1.3)
            output_tokens = int(len(response.text.split()) * 1.3)
        
        self.credit_tracker.update(
            model=model,
            input_tokens=int(input_tokens),
            output_tokens=int(output_tokens)
        )
        
        content = response.text.strip()
        
        # Clean markdown
        if content.startswith('```'):
            lines = content.split('\n')
            start_idx = 0
            end_idx = len(lines)
            for i, line in enumerate(lines):
                if line.strip().startswith('```'):
                    if start_idx == 0:
                        start_idx = i + 1
                    else:
                        end_idx = i
                        break
            content = '\n'.join(lines[start_idx:end_idx])
        
        # Parse JSON
        parsed = json.loads(content)
        if isinstance(parsed, dict):
            if 'assessments' in parsed:
                results_data = parsed['assessments']
            elif 'results' in parsed:
                results_data = parsed['results']
            else:
                results_data = next(v for v in parsed.values() if isinstance(v, list))
        else:
            results_data = parsed
        
        # Convert to AssessmentResult objects
        assessment_results = []
        for item_data in results_data:
            rank = int(item_data.get("rank", 0))
            sources = item_data.get("sources", [])
            reasoning = item_data.get("reasoning", "")
            
            if rank > 0 and len(sources) == 0:
                rank = 0
                reasoning += " [Rank lowered to 0: no sources]"
            
            assessment_results.append(AssessmentResult(
                software=item_data.get("software", ""),
                method=item_data.get("method", ""),
                rank=rank,
                reasoning=reasoning,
                sources=sources,
                llm_provider=f"google_{model.replace('models/', '')}",
                input_tokens=int(input_tokens) // len(batch_items),
                output_tokens=int(output_tokens) // len(batch_items)
            ))
        
        return assessment_results
        
    except Exception as e:
        print(f"  ERROR in Google batch assessment: {e}")
        return []

# Add methods to class
SoftwareMethodAssessor.assess_batch_with_openai = assess_batch_with_openai
SoftwareMethodAssessor.assess_batch_with_claude = assess_batch_with_claude
SoftwareMethodAssessor.assess_batch_with_google = assess_batch_with_google

print("✓ LLM assessment methods added")


✓ LLM assessment methods added


In [8]:
# =============================================================================
# Cell 8
# BATCH CREATION
# =============================================================================

def create_batches(self, software_list: List[str], method_list: List[str],
                  strategy: str = "by_software", batch_size: int = 50) -> List[List[Tuple[str, str]]]:
    """
    Create batches of (software, method) pairs
    
    Args:
        software_list: List of software names
        method_list: List of methods
        strategy: Batching strategy ('by_software', 'by_method', 'mixed', 'fixed_size')
        batch_size: Size for fixed_size batching
    
    Returns:
        List of batches
    """
    batches = []
    
    if strategy == "by_software":
        for software in software_list:
            batch = [(software, method) for method in method_list]
            batches.append(batch)
    
    elif strategy == "by_method":
        for method in method_list:
            batch = [(software, method) for software in software_list]
            batches.append(batch)
    
    elif strategy == "mixed":
        for i, software in enumerate(software_list[:len(software_list)//2 + 1]):
            batch = [(software, method) for method in method_list]
            batches.append(batch)
        for method in method_list:
            batch = [(software, method) for software in software_list[len(software_list)//2 + 1:]]
            if batch:
                batches.append(batch)
    
    elif strategy == "fixed_size":
        all_items = [(sw, method) for sw in software_list for method in method_list]
        for i in range(0, len(all_items), batch_size):
            batch = all_items[i:i + batch_size]
            batches.append(batch)
    
    return batches

SoftwareMethodAssessor.create_batches = create_batches

print("✓ Batch creation method added")


✓ Batch creation method added


In [9]:
# =============================================================================
# Cell 9
# GAP ANALYSIS - FIND MISSING PAIRS (SUPPORTS CSV AND JSON)
# =============================================================================
# =============================================================================
# CONVERT WIDE FORMAT TO LONG FORMAT (WITH DELIMITER DETECTION)
# =============================================================================

def convert_wide_to_long_format(csv_file: str, 
                                software_col: str = 'Name',
                                skip_cols: List[str] = None,
                                delimiter: str = None) -> pd.DataFrame:
    """
    Convert wide format CSV (software × methods matrix) to long format
    
    Args:
        csv_file: Path to wide format CSV
        software_col: Column containing software names
        skip_cols: Columns to skip (non-method columns)
        delimiter: CSV delimiter (auto-detect if None)
    
    Returns:
        DataFrame with columns: software, method, rank
    """
    print(f"\n{'='*70}")
    print(f"CONVERTING WIDE FORMAT TO LONG FORMAT")
    print(f"{'='*70}")
    
    # Auto-detect delimiter if not provided
    if delimiter is None:
        with open(csv_file, 'r', encoding='utf-8-sig') as f:
            first_line = f.readline()
            if ';' in first_line and first_line.count(';') > first_line.count(','):
                delimiter = ';'
            else:
                delimiter = ','
        print(f"Auto-detected delimiter: '{delimiter}'")
    
    # Load CSV with correct delimiter and handle decimal comma
    try:
        df = pd.read_csv(csv_file, sep=delimiter, encoding='utf-8-sig', on_bad_lines='skip')
    except Exception as e:
        print(f"⚠️  Error with utf-8-sig, trying utf-8: {e}")
        df = pd.read_csv(csv_file, sep=delimiter, encoding='utf-8', on_bad_lines='skip')
    
    print(f"Loaded CSV: {df.shape[0]} rows × {df.shape[1]} columns")
    
    # Default columns to skip (metadata, not methods)
    if skip_cols is None:
        skip_cols = [
            'Name', 'Include?', 'Vendor', 'Type', 'Coverage', 
            'Type of modelling', 'OSMM Score', 'OSMM - product maturity',
            'OSMM - product maturity - API (+1)',
            'OSMM - product maturity - Implementeringseksempel(+1)',
            'Unnamed: 0'  # Index column if present
        ]
    
    # Also skip any column that starts with OSMM or contains metadata keywords
    additional_skip = [col for col in df.columns 
                      if any(keyword in col.lower() for keyword in 
                             ['osmm', 'unnamed', 'include', 'vendor', 'type', 'coverage', 'modelling'])]
    skip_cols.extend(additional_skip)
    skip_cols = list(set(skip_cols))  # Remove duplicates
    
    # Identify method columns (all columns except metadata)
    method_cols = [col for col in df.columns 
                   if col not in skip_cols and col != software_col]
    
    print(f"Software column: '{software_col}'")
    print(f"Identified {len(method_cols)} method columns")
    print(f"Skipped {len(skip_cols)} metadata columns")
    
    # Convert to long format
    long_data = []
    
    for idx, row in df.iterrows():
        software = row[software_col]
        
        for method in method_cols:
            rank = row[method]
            
            # Skip NaN values (missing assessments)
            if pd.notna(rank):
                # Handle European decimal format (comma instead of dot)
                if isinstance(rank, str):
                    rank = rank.replace(',', '.')
                
                # Convert rank to int if possible
                try:
                    rank = int(float(rank))
                except:
                    # If conversion fails, skip this entry
                    continue
                
                long_data.append({
                    'software': software,
                    'method': method,
                    'rank': rank
                })
    
    long_df = pd.DataFrame(long_data)
    
    print(f"\n✓ Converted to long format:")
    print(f"  Total pairs: {len(long_df)}")
    print(f"  Unique software: {long_df['software'].nunique()}")
    print(f"  Unique methods: {long_df['method'].nunique()}")
    
    # Show sample
    if len(long_df) > 0:
        print(f"\nSample entries:")
        print(long_df.head(3).to_string(index=False))
    
    print(f"{'='*70}\n")
    
    return long_df

print("✓ Wide-to-long format converter defined (with delimiter detection)")


# =============================================================================
# GAP ANALYSIS - FIND MISSING PAIRS (SUPPORTS CSV FORMATS AND JSON)
# =============================================================================

def identify_missing_pairs(self, software_list: List[str], method_list: List[str],
                          existing_results_file: str = None,
                          software_col: str = 'Name',
                          is_wide_format: bool = True) -> List[Tuple[str, str]]:
    """
    Identify software-method pairs that haven't been assessed yet
    
    Args:
        software_list: Complete list of software
        method_list: Complete list of methods
        existing_results_file: Path to existing results (CSV or JSON)
        software_col: Column name for software in wide format CSV
        is_wide_format: If True, treats CSV as wide format (software × methods matrix)
    
    Returns:
        List of missing (software, method) pairs
    """
    print(f"\n{'='*70}")
    print(f"IDENTIFYING MISSING PAIRS")
    print(f"{'='*70}")
    
    # Expected pairs
    expected_pairs = set((sw, m) for sw in software_list for m in method_list)
    print(f"Expected total pairs: {len(expected_pairs)}")
    
    # Load existing if provided
    if existing_results_file and Path(existing_results_file).exists():
        file_ext = Path(existing_results_file).suffix.lower()
        
        if file_ext == '.csv':
            print(f"Loading existing results from CSV: {existing_results_file}")
            
            if is_wide_format:
                # Convert wide format to long format
                long_df = convert_wide_to_long_format(
                    existing_results_file, 
                    software_col=software_col
                )
                existing_pairs = set(zip(long_df['software'], long_df['method']))
            else:
                # Load as long format (software, method columns)
                df = pd.read_csv(existing_results_file)
                
                # Try common column name variations
                sw_col = None
                method_col = None
                
                for col in df.columns:
                    col_lower = col.lower()
                    if col_lower in ['software', 'sw', 'software_name', 'tool']:
                        sw_col = col
                    if col_lower in ['method', 'methods', 'method_name', 'technique']:
                        method_col = col
                
                if sw_col is None or method_col is None:
                    print(f"⚠️  WARNING: Could not identify software/method columns")
                    print(f"Available columns: {list(df.columns)}")
                    sw_col = df.columns[0]
                    method_col = df.columns[1]
                    print(f"Using: software='{sw_col}', method='{method_col}'")
                
                existing_pairs = set(zip(df[sw_col], df[method_col]))
            
        elif file_ext == '.json':
            print(f"Loading existing results from JSON: {existing_results_file}")
            with open(existing_results_file, 'r') as f:
                existing_results = json.load(f)
            
            existing_pairs = set((r['software'], r['method']) for r in existing_results)
        
        else:
            print(f"⚠️  WARNING: Unsupported file format: {file_ext}")
            print(f"Supported formats: .csv, .json")
            existing_pairs = set()
        
        print(f"Existing pairs: {len(existing_pairs)}")
        
        # Find missing
        missing_pairs = expected_pairs - existing_pairs
        print(f"Missing pairs: {len(missing_pairs)}")
        
        if len(missing_pairs) > 0:
            # Summary
            missing_software = set(sw for sw, _ in missing_pairs)
            missing_methods = set(m for _, m in missing_pairs)
            print(f"\nMissing data for:")
            print(f"  Software: {len(missing_software)}")
            if len(missing_software) <= 10:
                print(f"    {', '.join(sorted(missing_software))}")
            else:
                print(f"    {', '.join(sorted(missing_software)[:5])}... and {len(missing_software)-5} more")
            
            print(f"  Methods: {len(missing_methods)}")
            if len(missing_methods) <= 10:
                print(f"    {', '.join(sorted(missing_methods))}")
            else:
                print(f"    {', '.join(sorted(missing_methods)[:5])}... and {len(missing_methods)-5} more")
    else:
        if existing_results_file:
            print(f"⚠️  File not found: {existing_results_file}")
        print(f"No existing results - all pairs need assessment")
        missing_pairs = expected_pairs
    
    print(f"{'='*70}\n")
    
    return list(missing_pairs)

SoftwareMethodAssessor.identify_missing_pairs = identify_missing_pairs

print("✓ Gap analysis method added (supports wide/long CSV and JSON)")


✓ Wide-to-long format converter defined (with delimiter detection)
✓ Gap analysis method added (supports wide/long CSV and JSON)


In [10]:
# =============================================================================
# Cell 10a
# UNIFIED ASSESSMENT METHOD - WITH GAP REVIEW OPTION
# =============================================================================

def assess_multiple_batched(self, software_list: List[str], method_list: List[str],
                           existing_results_file: str = None,
                           batch_strategy: str = "fixed_size",
                           batch_size: int = 20,
                           use_openai: bool = True,
                           use_claude: bool = True,
                           use_google: bool = True,
                           openai_model: str = None,
                           claude_model: str = "claude-3-5-haiku-20241022",
                           google_model: str = "models/gemini-2.0-flash",
                           checkpoint_every: int = 50,
                           review_gaps: bool = True,
                           debug: bool = False) -> List[ConsensusResult]:
    """
    Unified assessment method - handles both new and incremental updates
    
    Args:
        software_list: List of software to assess
        method_list: List of methods to assess
        existing_results_file: Path to existing results (if updating)
        batch_strategy: Batching strategy
        batch_size: Size of batches
        use_openai, use_claude, use_google: Which LLMs to use
        openai_model, claude_model, google_model: Model names
        checkpoint_every: Save checkpoint every N batches
        review_gaps: If True, pause after gap analysis for manual review
        debug: Debug mode
    
    Returns:
        List of ConsensusResult objects
    """
    
    # Step 1: Identify missing pairs
    missing_pairs = self.identify_missing_pairs(software_list, method_list, existing_results_file)
    
    if len(missing_pairs) == 0:
        print("✓ No missing pairs - all assessments complete!")
        if existing_results_file:
            with open(existing_results_file, 'r') as f:
                existing_data = json.load(f)
            return [ConsensusResult(**r) for r in existing_data]
        else:
            return []
    
    # ========================================================================
    # INTERCEPTION POINT - Review and filter gaps
    # ========================================================================
    if review_gaps:
        print(f"\n{'='*70}")
        print(f"GAP REVIEW - INTERCEPTION POINT")
        print(f"{'='*70}")
        print(f"Found {len(missing_pairs)} missing pairs")
        print(f"\nSample pairs:")
        for i, (sw, m) in enumerate(missing_pairs[:10], 1):
            print(f"  {i}. {sw} × {m}")
        if len(missing_pairs) > 10:
            print(f"  ... and {len(missing_pairs) - 10} more")
        
        # Export for review
        review_file = output_dir / f"gaps_to_review_{timestamp}.csv"
        gaps_df = pd.DataFrame(missing_pairs, columns=['software', 'method'])
        gaps_df.to_csv(review_file, index=False)
        print(f"\n✓ Gaps exported to: {review_file}")
        print(f"\nYou can now:")
        print(f"  1. Review the gaps in the CSV file")
        print(f"  2. Edit/filter as needed")
        print(f"  3. Continue execution in the next cell with filtered pairs")
        print(f"\n⚠️  EXECUTION PAUSED - Run next cell to continue")
        print(f"{'='*70}\n")
        
        # Store for later use
        self._pending_missing_pairs = missing_pairs
        self._pending_params = {
            'software_list': software_list,
            'method_list': method_list,
            'existing_results_file': existing_results_file,
            'batch_strategy': batch_strategy,
            'batch_size': batch_size,
            'use_openai': use_openai,
            'use_claude': use_claude,
            'use_google': use_google,
            'openai_model': openai_model,
            'claude_model': claude_model,
            'google_model': google_model,
            'checkpoint_every': checkpoint_every,
            'debug': debug
        }
        
        return []  # Return empty, will continue in next cell
    
    # If not reviewing, continue with assessment
    return self._execute_batch_assessment(
        missing_pairs=missing_pairs,
        existing_results_file=existing_results_file,
        batch_strategy=batch_strategy,
        batch_size=batch_size,
        use_openai=use_openai,
        use_claude=use_claude,
        use_google=use_google,
        openai_model=openai_model,
        claude_model=claude_model,
        google_model=google_model,
        checkpoint_every=checkpoint_every,
        debug=debug
    )

SoftwareMethodAssessor.assess_multiple_batched = assess_multiple_batched

print("✓ Unified assessment method with gap review added")


✓ Unified assessment method with gap review added


In [11]:
# =============================================================================
# Cell 10b
# INTERNAL ASSESSMENT EXECUTION (after gap review)
# =============================================================================

def _execute_batch_assessment(self, missing_pairs: List[Tuple[str, str]],
                              existing_results_file: str = None,
                              batch_strategy: str = "fixed_size",
                              batch_size: int = 20,
                              use_openai: bool = True,
                              use_claude: bool = True,
                              use_google: bool = True,
                              openai_model: str = None,
                              claude_model: str = "claude-3-5-haiku-20241022",
                              google_model: str = "models/gemini-2.0-flash",
                              checkpoint_every: int = 50,
                              debug: bool = False) -> List[ConsensusResult]:
    """Internal method to execute batch assessment after gap review"""
    
    print(f"\n{'='*70}")
    print(f"BATCH ASSESSMENT MODE")
    print(f"{'='*70}")
    print(f"Pairs to assess: {len(missing_pairs)}")
    print(f"Strategy: {batch_strategy}")
    print(f"Batch size: {batch_size}")
    print(f"LLMs: OpenAI={use_openai}, Claude={use_claude}, Google={use_google}")
    
    # Create batches from missing pairs
    if batch_strategy == "fixed_size":
        batches = [missing_pairs[i:i + batch_size] for i in range(0, len(missing_pairs), batch_size)]
    else:
        # Extract unique software and methods from missing pairs
        missing_software = sorted(set(sw for sw, _ in missing_pairs))
        missing_methods = sorted(set(m for _, m in missing_pairs))
        batches = self.create_batches(missing_software, missing_methods, batch_strategy, batch_size)
    
    print(f"\nCreated {len(batches)} batches")
    
    # Assess batches
    all_assessments = {}
    failed_batches = []
    
    print(f"\n{'-'*70}")
    print(f"Processing batches...")
    print(f"{'-'*70}")
    
    for batch_idx, batch in enumerate(batches, 1):
        print(f"\n[Batch {batch_idx}/{len(batches)}] {len(batch)} items", flush=True)
        
        batch_results = []
        
        if use_openai and self.openai_client:
            try:
                print(f"  OpenAI...", end='', flush=True)
                results = self.assess_batch_with_openai(batch, openai_model, debug)
                batch_results.extend(results)
                print(f" ✓ {len(results)}", flush=True)
            except Exception as e:
                print(f" ✗ {str(e)[:50]}", flush=True)
                failed_batches.append(('openai', batch_idx))
            time.sleep(2)
        
        if use_claude and self.anthropic_client:
            try:
                print(f"  Claude...", end='', flush=True)
                results = self.assess_batch_with_claude(batch, claude_model, debug)
                batch_results.extend(results)
                print(f" ✓ {len(results)}", flush=True)
            except Exception as e:
                print(f" ✗ {str(e)[:50]}", flush=True)
                failed_batches.append(('claude', batch_idx))
            time.sleep(2)
        
        if use_google and self.google_enabled:
            try:
                print(f"  Google...", end='', flush=True)
                results = self.assess_batch_with_google(batch, google_model, debug)
                batch_results.extend(results)
                print(f" ✓ {len(results)}", flush=True)
            except Exception as e:
                print(f" ✗ {str(e)[:50]}", flush=True)
                failed_batches.append(('google', batch_idx))
            time.sleep(2)
        
        # Store results
        for result in batch_results:
            key = (result.software, result.method)
            if key not in all_assessments:
                all_assessments[key] = []
            all_assessments[key].append(result)
        
        # Progress
        if batch_idx % 10 == 0:
            stats = self.credit_tracker.get_stats()
            print(f"  Progress: ${stats['total_cost']:.2f} | {len(all_assessments)} unique pairs")
        
        # Checkpoint
        if batch_idx % checkpoint_every == 0:
            checkpoint_file = output_dir / f"checkpoint_{batch_idx}_{timestamp}.pkl"
            with open(checkpoint_file, 'wb') as f:
                pickle.dump(all_assessments, f)
            print(f"  💾 Checkpoint saved: {checkpoint_file.name}")
    
    # Create consensus results
    print(f"\n{'-'*70}")
    print(f"Creating consensus results...")
    print(f"{'-'*70}")
    
    new_consensus_results = []
    
    for (software, method), assessments in all_assessments.items():
        # Remove duplicates by provider
        by_provider = {}
        for a in assessments:
            if a.llm_provider not in by_provider:
                by_provider[a.llm_provider] = a
        assessments = list(by_provider.values())
        
        if len(assessments) == 0:
            continue
        
        ranks = [a.rank for a in assessments]
        confidence, agreement_level = self.calculate_confidence(ranks)
        rank_counts = Counter(ranks)
        final_rank = rank_counts.most_common(1)[0][0]
        
        new_consensus_results.append(ConsensusResult(
            software=software,
            method=method,
            final_rank=final_rank,
            confidence=confidence,
            individual_ranks={a.llm_provider: a.rank for a in assessments},
            individual_reasoning={a.llm_provider: a.reasoning for a in assessments},
            individual_sources={a.llm_provider: a.sources for a in assessments},
            agreement_level=agreement_level,
            total_tokens=sum(a.input_tokens + a.output_tokens for a in assessments),
            total_cost=0.0
        ))
    
    # Merge with existing if applicable
    if existing_results_file and Path(existing_results_file).exists():
        print(f"\nMerging with existing results from: {existing_results_file}")
        with open(existing_results_file, 'r') as f:
            existing_data = json.load(f)
        
        existing_results = [ConsensusResult(**r) for r in existing_data]
        
        # Combine
        all_results = existing_results + new_consensus_results
        print(f"  Existing: {len(existing_results)}")
        print(f"  New: {len(new_consensus_results)}")
        print(f"  Total: {len(all_results)}")
    else:
        all_results = new_consensus_results
    
    print(f"\n✓ Completed {len(new_consensus_results)} new assessments")
    print(f"{'='*70}\n")
    
    return all_results

SoftwareMethodAssessor._execute_batch_assessment = _execute_batch_assessment

print("✓ Internal batch assessment executor added")


✓ Internal batch assessment executor added


In [12]:
# =============================================================================
# Cell 10c
# CONTINUE ASSESSMENT AFTER GAP REVIEW
# =============================================================================

def continue_assessment_after_review(self, 
                                     filtered_pairs: List[Tuple[str, str]] = None,
                                     filtered_csv: str = None) -> List[ConsensusResult]:
    """
    Continue assessment after reviewing and optionally filtering gaps
    
    Args:
        filtered_pairs: Manually filtered list of (software, method) tuples
        filtered_csv: Path to CSV with filtered pairs (columns: 'software', 'method')
    
    Returns:
        List of ConsensusResult objects
    """
    if not hasattr(self, '_pending_missing_pairs'):
        print("❌ No pending assessment found. Run assess_multiple_batched() first with review_gaps=True")
        return []
    
    # Determine which pairs to use
    if filtered_csv:
        print(f"Loading filtered pairs from: {filtered_csv}")
        df = pd.read_csv(filtered_csv)
        missing_pairs = list(zip(df['software'], df['method']))
        print(f"✓ Loaded {len(missing_pairs)} filtered pairs from CSV")
    elif filtered_pairs:
        missing_pairs = filtered_pairs
        print(f"✓ Using {len(missing_pairs)} manually filtered pairs")
    else:
        missing_pairs = self._pending_missing_pairs
        print(f"✓ Using all {len(missing_pairs)} original pairs (no filtering)")
    
    if len(missing_pairs) == 0:
        print("⚠️  No pairs to assess after filtering")
        return []
    
    print(f"\n{'='*70}")
    print(f"CONTINUING ASSESSMENT")
    print(f"{'='*70}")
    print(f"Pairs to assess: {len(missing_pairs)}")
    
    # Get stored parameters
    params = self._pending_params
    
    # Execute assessment
    results = self._execute_batch_assessment(
        missing_pairs=missing_pairs,
        existing_results_file=params['existing_results_file'],
        batch_strategy=params['batch_strategy'],
        batch_size=params['batch_size'],
        use_openai=params['use_openai'],
        use_claude=params['use_claude'],
        use_google=params['use_google'],
        openai_model=params['openai_model'],
        claude_model=params['claude_model'],
        google_model=params['google_model'],
        checkpoint_every=params['checkpoint_every'],
        debug=params['debug']
    )
    
    # Clean up stored data
    del self._pending_missing_pairs
    del self._pending_params
    
    return results

SoftwareMethodAssessor.continue_assessment_after_review = continue_assessment_after_review

print("✓ Continue assessment method added")


✓ Continue assessment method added


In [13]:
# =============================================================================
# Cell 11
# RESULT MERGER - MERGE MULTIPLE RESULT FILES
# =============================================================================

def merge_assessment_results(self, *result_files: str, output_file: str = "merged_results.json") -> List[ConsensusResult]:
    """
    Merge multiple assessment result JSON files
    
    Args:
        *result_files: Paths to result JSON files
        output_file: Output merged file path
    
    Returns:
        List of merged ConsensusResult objects
    """
    print(f"\n{'='*70}")
    print(f"MERGING ASSESSMENT RESULTS")
    print(f"{'='*70}")
    print(f"Input files: {len(result_files)}")
    
    merged_data = {}
    
    for file_idx, file_path in enumerate(result_files, 1):
        print(f"\nProcessing file {file_idx}/{len(result_files)}: {file_path}")
        
        try:
            with open(file_path, 'r') as f:
                results = json.load(f)
            
            print(f"  Loaded {len(results)} assessments")
            
            for result in results:
                software = result['software']
                method = result['method']
                key = (software, method)
                
                if key not in merged_data:
                    merged_data[key] = result
                else:
                    # Merge: combine all LLM assessments
                    merged_data[key]['individual_ranks'].update(result['individual_ranks'])
                    merged_data[key]['individual_reasoning'].update(result['individual_reasoning'])
                    merged_data[key]['individual_sources'].update(result['individual_sources'])
                    merged_data[key]['total_tokens'] += result['total_tokens']
                    merged_data[key]['total_cost'] += result['total_cost']
            
        except Exception as e:
            print(f"  ERROR loading {file_path}: {e}")
            continue
    
    # Recalculate consensus
    print(f"\nRecalculating consensus for merged results...")
    merged_results_list = list(merged_data.values())
    
    for result in merged_results_list:
        ranks = list(result['individual_ranks'].values())
        rank_counts = Counter(ranks)
        result['final_rank'] = rank_counts.most_common(1)[0][0]
        
        most_common_count = rank_counts.most_common(1)[0][1]
        result['confidence'] = most_common_count / len(ranks)
        
        if result['confidence'] == 1.0:
            result['agreement_level'] = "perfect_agreement"
        elif result['confidence'] >= 0.75:
            result['agreement_level'] = "strong_agreement"
        elif result['confidence'] >= 0.5:
            result['agreement_level'] = "moderate_agreement"
        else:
            result['agreement_level'] = "weak_agreement"
    
    # Save
    with open(output_file, 'w') as f:
        json.dump(merged_results_list, f, indent=2)
    
    print(f"\n✓ Merged results saved to: {output_file}")
    print(f"{'='*70}\n")
    
    consensus_results = [ConsensusResult(**r) for r in merged_results_list]
    return consensus_results

SoftwareMethodAssessor.merge_assessment_results = merge_assessment_results

print("✓ Result merger added")


✓ Result merger added


In [14]:
# =============================================================================
# Cell 11b 
# CONVERT LONG FORMAT TO WIDE FORMAT (MATRIX)
# =============================================================================

def convert_long_to_wide_format(results: List[ConsensusResult], 
                                output_file: str = None) -> pd.DataFrame:
    """
    Convert long format results to wide format (software × methods matrix)
    
    Args:
        results: List of ConsensusResult objects
        output_file: Optional path to save CSV
    
    Returns:
        DataFrame in wide format (software as rows, methods as columns)
    """
    print(f"\n{'='*70}")
    print(f"CONVERTING TO WIDE FORMAT (MATRIX)")
    print(f"{'='*70}")
    
    # Create long format first
    long_data = []
    for r in results:
        long_data.append({
            'software': r.software,
            'method': r.method,
            'rank': r.final_rank
        })
    
    long_df = pd.DataFrame(long_data)
    print(f"Long format: {len(long_df)} rows")
    
    # Pivot to wide format
    wide_df = long_df.pivot(index='software', columns='method', values='rank')
    
    # Reset index to make software a column
    wide_df = wide_df.reset_index()
    wide_df = wide_df.rename(columns={'software': 'Name'})
    
    print(f"Wide format: {wide_df.shape[0]} rows × {wide_df.shape[1]} columns")
    print(f"  Software: {wide_df.shape[0]}")
    print(f"  Methods: {wide_df.shape[1] - 1}")  # -1 for Name column
    
    # Save if requested
    if output_file:
        wide_df.to_csv(output_file, index=False)
        print(f"\n✓ Wide format saved to: {output_file}")
    
    print(f"{'='*70}\n")
    
    return wide_df

SoftwareMethodAssessor.convert_long_to_wide_format = convert_long_to_wide_format

print("✓ Long-to-wide format converter added")

# =============================================================================
# MERGE NEW RESULTS WITH EXISTING METADATA (WITH DELIMITER DETECTION)
# =============================================================================

def merge_with_existing_metadata(new_results_wide: pd.DataFrame,
                                 existing_file: str,
                                 software_col: str = 'Name',
                                 metadata_cols: List[str] = None,
                                 delimiter: str = None) -> pd.DataFrame:
    """
    Merge new assessment results with metadata from existing file
    
    Args:
        new_results_wide: New results in wide format
        existing_file: Path to existing CSV with metadata
        software_col: Software column name
        metadata_cols: Metadata columns to keep (if None, auto-detect)
        delimiter: CSV delimiter (auto-detect if None)
    
    Returns:
        Merged DataFrame with metadata + new assessments
    """
    print(f"\n{'='*70}")
    print(f"MERGING WITH EXISTING METADATA")
    print(f"{'='*70}")
    
    # Auto-detect delimiter if not provided
    if delimiter is None:
        with open(existing_file, 'r', encoding='utf-8-sig') as f:
            first_line = f.readline()
            if ';' in first_line and first_line.count(';') > first_line.count(','):
                delimiter = ';'
            else:
                delimiter = ','
        print(f"Auto-detected delimiter: '{delimiter}'")
    
    # Load existing file
    try:
        existing_df = pd.read_csv(existing_file, sep=delimiter, encoding='utf-8-sig', on_bad_lines='skip')
    except:
        existing_df = pd.read_csv(existing_file, sep=delimiter, encoding='utf-8', on_bad_lines='skip')
    
    print(f"Loaded existing file: {existing_df.shape}")
    
    # Auto-detect metadata columns if not provided
    if metadata_cols is None:
        metadata_cols = [
            'Include?', 'Vendor', 'Type', 'Coverage', 
            'Type of modelling', 'OSMM Score', 'OSMM - product maturity',
            'OSMM - product maturity - API (+1)',
            'OSMM - product maturity - Implementeringseksempel(+1)'
        ]
        # Keep only columns that exist
        metadata_cols = [col for col in metadata_cols if col in existing_df.columns]
        
        # Add any other columns that look like metadata
        additional_metadata = [col for col in existing_df.columns 
                              if any(keyword in col.lower() for keyword in 
                                     ['osmm', 'include', 'vendor', 'type', 'coverage', 'modelling'])]
        metadata_cols.extend(additional_metadata)
        metadata_cols = list(set(metadata_cols))  # Remove duplicates
    
    print(f"Metadata columns: {len(metadata_cols)} columns")
    
    # Extract metadata
    metadata_df = existing_df[[software_col] + metadata_cols].copy()
    metadata_df = metadata_df.rename(columns={software_col: 'Name'})
    
    # Merge with new results
    merged_df = metadata_df.merge(
        new_results_wide,
        on='Name',
        how='outer',
        suffixes=('_old', '_new')
    )
    
    # Update old values with new ones where available
    for col in new_results_wide.columns:
        if col != 'Name' and f'{col}_new' in merged_df.columns:
            merged_df[col] = merged_df[f'{col}_new'].combine_first(merged_df.get(f'{col}_old', merged_df[f'{col}_new']))
            # Drop temporary columns
            if f'{col}_new' in merged_df.columns:
                merged_df = merged_df.drop(columns=[f'{col}_new'])
            if f'{col}_old' in merged_df.columns:
                merged_df = merged_df.drop(columns=[f'{col}_old'])
    
    print(f"Merged result: {merged_df.shape}")
    print(f"{'='*70}\n")
    
    return merged_df

SoftwareMethodAssessor.merge_with_existing_metadata = merge_with_existing_metadata

print("✓ Metadata merger added (with delimiter detection)")



✓ Long-to-wide format converter added
✓ Metadata merger added (with delimiter detection)


In [25]:
# =============================================================================
## Cell 12
# LOAD SOFTWARE AND METHOD LISTS
# =============================================================================

def load_method_list(json_file: str = "method_variant_groups.json") -> list:
    """Load canonical method names from variant groups JSON"""
    json_path = Path(SAVE_DIR) / json_file
    
    if not json_path.exists():
        raise FileNotFoundError(f"Method groups file not found: {json_file}")
    
    with open(json_path, 'r', encoding='utf-8') as f:
        method_groups = json.load(f)
    
    method_list = list(method_groups.keys())
    print(f"✓ Loaded {len(method_list)} methods from: {json_file}")
    
    return method_list

software_list_all = ['Power Factory Digisilent', 
                     'DINIS', 'ERACS', 'Distribution Network Analysis', 'IPSA', 'Power World', 
                     'PSS/E', 'PSSE/SINCAL', 'SKM Power Tools', 'OpenDSS', 'Matlab & Simulink', 'DYMOLA', 'MathPower',
                     'RelyPES', 'GridLAB-D', 'PyPSA (Python for Power System Analysis)', 'TARA', 'PyPower/Pandapower', 'GridCal Sk', 'MatDyn', 
                     'NEPLAN', 'PSAT', 'CYMEDIST', 'Synergi Electric', 'Dynawo', 'OpenModellica', 
                     'Sienna', 'POWSYBL', 'Hitachi Network Manager', 'Spectrum Power', 'CIMPLICITY Scada', 'eTerra', 'Netbas', 'Trimble NIS', 'GAMS','Promaps','ETAP','OpenModelica',
#added from IEEE
                    'MARS' , 'PLEXOS', 'Gridview', 'ANTARES', 'TRELSS', 'SERVM', 'CORAL', 'Aristo', 'REMARK', 'BID3','PROMOD IV'
]


# Load methods from your JSON file
method_list_all = load_method_list("method_variant_groups.json")

print(f"✓ Loaded {len(software_list_all)} software")
print(f"✓ Loaded {len(method_list_all)} methods")
print(f"✓ Total pairs to assess: {len(software_list_all) * len(method_list_all)}")


✓ Loaded 322 methods from: method_variant_groups.json
✓ Loaded 49 software
✓ Loaded 322 methods
✓ Total pairs to assess: 15778


In [26]:
# =============================================================================
# EXECUTE ASSESSMENT - WITH GAP REVIEW OPTION
# =============================================================================

# Initialize assessor
assessor = SoftwareMethodAssessor(use_config=True, timeout=180)

# Your existing file path
existing_file = r"C:\git_repos\Literature-search-and-analysis\software_analysis_output\software_methods_FINAL_COMPLETE_20251213_085154.csv"

# Step 1: Run gap analysis (will pause for review)
results = assessor.assess_multiple_batched(
    software_list=software_list_all,
    method_list=method_list_all,
    existing_results_file=existing_file,
    batch_strategy="fixed_size",
    batch_size=20,
    use_openai=True,
    use_claude=False,
    use_google=True,
    openai_model='gpt-4o-mini',
    google_model='models/gemini-2.0-flash',
    checkpoint_every=50,
    review_gaps=True,
    debug=False
)



IDENTIFYING MISSING PAIRS
Expected total pairs: 15778
Loading existing results from CSV: C:\git_repos\Literature-search-and-analysis\software_analysis_output\software_methods_FINAL_COMPLETE_20251213_085154.csv

CONVERTING WIDE FORMAT TO LONG FORMAT
Auto-detected delimiter: ';'
Loaded CSV: 39 rows × 268 columns
Software column: 'Name'
Identified 247 method columns
Skipped 22 metadata columns

✓ Converted to long format:
  Total pairs: 9251
  Unique software: 39
  Unique methods: 247

Sample entries:
                software                                  method  rank
Power Factory Digisilent                     power flow analysis     3
Power Factory Digisilent security-constrained optimal power flow     2
Power Factory Digisilent    security-constrained unit commitment     3

Existing pairs: 9251
Missing pairs: 8114

Missing data for:
  Software: 49
    ANTARES, Aristo, BID3, CIMPLICITY Scada, CORAL... and 44 more
  Methods: 322
    actor-critic methods, adaptive modulation, adaptiv

Review the gap file and modify if any should be removed

In [27]:
# Load, filter, and assess in one go
gaps_df = pd.read_csv(output_dir / f"gaps_to_review_{timestamp}.csv")



In [30]:
# count each time each software is found in the df
software_counts = gaps_df['software'].value_counts()
print(software_counts)
method_counts= gaps_df['method'].value_counts()


software
ANTARES                                     301
BID3                                        301
CORAL                                       301
Promaps                                     301
Gridview                                    301
PLEXOS                                      301
OpenModelica                                301
REMARK                                      301
TRELSS                                      301
MARS                                        301
SERVM                                       301
Aristo                                      301
PROMOD IV                                   301
Sienna                                      135
Distribution Network Analysis               135
PyPSA (Python for Power System Analysis)    114
DYMOLA                                      108
CIMPLICITY Scada                             93
GridCal Sk                                   93
Synergi Electric                             93
PyPower/Pandapower             

In [ ]:
# Quick filters
gaps_df = gaps_df[~gaps_df['software'].isin(['DINIS',
                                             'Distribution Network Analysis - ETAP','ETAP',
                                             'MatDyn','RelyPES','CYMEDIST','DINID','OpenModellica','SKM Power Tools','Gridlab-D','IPSA','ERACS','OpenModellica',
                                             'Promaps'])]
gaps_df = gaps_df[~gaps_df['method'].isin(['zero-forcing',
                                           'wavelet transform dwt',
                                           'genetic programming', 
                                           'time-frequency analysis',
                                           'experience replay',
                                           'point estimate method',
                                           'temporal difference learning',
                                           'meta-learning',
                                           'whale optimization algorithm',
                                           'quadrature amplitude modulation',
                                           'quadrature phase shift keying',
                                           'phase shift keying',
                                           'successive interference cancellation',
                                           'self-interference',
                                           'non-orthogonal multiple access',
                                           'orthogonal frequency-division multiplexing',
                                           'multiple-input-multiple-output',
                                           'multi-user detection',
                                           'signal noise ratio',
                                           'space vector pulse width modulation',
                                           'frequency hopping',
                                           'approximate computing'
                                           ])]

print(f"Assessing {len(gaps_df)} filtered pairs")


Assessing 6544 filtered pairs


In [ ]:
# =============================================================================
# Cell 14
# CONTINUE ASSESSMENT (after reviewing gaps) - SAVE BOTH FORMATS
# =============================================================================

# Continue
filtered_pairs = list(zip(gaps_df['software'], gaps_df['method']))
results = assessor.continue_assessment_after_review(filtered_pairs=filtered_pairs)




# ============================================================================
# SAVE RESULTS IN MULTIPLE FORMATS
# ============================================================================

print(f"\n{'='*70}")
print(f"SAVING RESULTS")
print(f"{'='*70}")

# 1. Save JSON (long format)
json_file = output_dir / f"assessment_results_{timestamp}.json"
assessor.export_results(results, str(json_file))
print(f"✓ JSON (long format): {json_file}")

# 2. Save CSV long format
long_df = pd.DataFrame([{
    'software': r.software,
    'method': r.method,
    'final_rank': r.final_rank,
    'confidence': r.confidence,
    'agreement_level': r.agreement_level,
    'num_llms': len(r.individual_ranks)
} for r in results])

csv_long_file = output_dir / f"assessment_results_long_{timestamp}.csv"
long_df.to_csv(csv_long_file, index=False)
print(f"✓ CSV long format: {csv_long_file}")

# 3. Save CSV wide format (MATRIX - for your analysis)
csv_wide_file = output_dir / f"assessment_results_wide_{timestamp}.csv"
wide_df = assessor.convert_long_to_wide_format(results, str(csv_wide_file))
print(f"✓ CSV wide format (MATRIX): {csv_wide_file}")

# 4. Preview wide format
print(f"\nWide format preview:")
print(wide_df.head())
print(f"\nShape: {wide_df.shape[0]} software × {wide_df.shape[1]-1} methods")

# 5. Merge with existing metadata (optional)
if existing_file and Path(existing_file).exists():
    csv_merged_file = output_dir / f"assessment_results_MERGED_{timestamp}.csv"
    merged_df = assessor.merge_with_existing_metadata(
        wide_df, 
        existing_file,
        software_col='Name'
    )
    merged_df.to_csv(csv_merged_file, index=False)
    print(f"✓ CSV merged (with metadata): {csv_merged_file}")
    print(f"  Shape: {merged_df.shape[0]} software × {merged_df.shape[1]} total columns")
# Print summary
assessor.credit_tracker.print_summary()

print(f"\n{'='*70}")
print(f"ASSESSMENT COMPLETE")
print(f"{'='*70}")
print(f"✓ Total assessments: {len(results)}")
print(f"\nFiles saved:")
print(f"  1. JSON: {json_file.name}")
print(f"  2. CSV (long): {csv_long_file.name}")
print(f"  3. CSV (wide/matrix): {csv_wide_file.name}  ← USE THIS FOR ANALYSIS")
print(f"{'='*70}\n")


In [22]:
df_test=pd.read_csv("C:\git_repos\Literature-search-and-analysis\software_analysis_output\software_methods_FINAL_COMPLETE_20251213_085154.csv",sep=";")

In [24]:
print(df_test['Name'].to_list())

['Power Factory Digisilent', 'DINIS', 'ERACS', 'Distribution Network Analysis', 'IPSA', 'Power World', 'PSS/E', 'PSSE/SINCAL', 'SKM Power Tools', 'OpenDSS', 'Matlab & Simulink', 'DYMOLA', 'MathPower', 'RelyPES', 'GridLAB-D', 'PyPSA (Python for Power System Analysis)', 'TARA', 'PyPower/Pandapower', 'GridCal Sk', 'MatDyn', 'NEPLAN', 'PSAT', 'CYMEDIST', 'Synergi Electric', 'Dynawo', 'OpenModellica', 'Sienna', 'POWSYBL', 'Hitachi Network Manager', 'Spectrum Power', 'CIMPLICITY Scada', 'eTerra', 'Netbas', 'Trimble NIS', 'GAMS', 'Sum raw implementation score', 'Gjennomsnittlig score (MIS)', 'Maks teoretisk mulig score ', 'Maks teoretisk mulig score (Alle har implementert på nivå 3) ']
